In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
packageVersion("marginaleffects")

[1] ‘0.25.1’

In [4]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "GraduationPaper/RevisetoJournal/9991-MergedData_similarity.csv"))
dim(data)

[1] 317275    103

In [5]:
print(names(data))

  [1] "X"                                           
  [2] "work_id"                                     
  [3] "PublishedYear"                               
  [4] "Facility"                                    
  [5] "num_fac"                                     
  [6] "paper_type"                                  
  [7] "paper_language"                              
  [8] "novel_uzzi"                                  
  [9] "novel_uzzi_bin"                              
 [10] "num_fac_scientist"                           
 [11] "ratio_fac_scientist"                         
 [12] "bin_fac_scientist"                           
 [13] "text_fac_scientist"                          
 [14] "fac_scientist_team"                          
 [15] "num_leader"                                  
 [16] "ratio_leader"                                
 [17] "bin_leader"                                  
 [18] "fac_scientist_lead_relratio"                 
 [19] "fac_scientist_lead_num"                

In [6]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                               PublishedYear 
                                           0 
                                    Facility 
                                           0 
                                     num_fac 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                  novel_uzzi 
                                        2532 
                              novel_uzzi_bin 
                                           0 
                           num_fac_scientist 
                                           0 
                         ratio_fac_scientist 
                                           0 
                           bin_fac_scientist 
                                           0 
                          text_fac_scientist 
                                           0 
                          fac_scientist_team 
                                           0 
                                  num_leader 
                                           0 
                                ratio_leader 
                                           0 
                                  bin_leader 
                                           0 
                 fac_scientist_lead_relratio 
                                           0 
                      fac_scientist_lead_num 
                                           0 
                    fac_scientist_lead_ratio 
                                           0 
                      fac_scientist_lead_bin 
                                           0 
                     fac_scientist_lead_text 
                                           0 
                                      CoType 
                                           0 
                        CoType_Collaboration 
                                           0 
                        CoType_Participation 
                                           0 
                              CoType_Service 
                                           0 
                                lnnum_author 
                                           0 
                                  lnnum_inst 
                                           0 
                               lnnum_country 
                                           0 
                               international 
                                           0 
                             lnnum_reference 
                                           0 
                                 open_access 
                                           0 
                                 RaoStirling 
                                           0 
                                         SDG 
                                           0 
                               lntimescited5 
                                       14695 
                              lntimescited10 
                                       12880 
                             lntimescitedall 
                                           0 
                                 lnab_length 
                                           0 
                           lnmean_career_age 
                                           0 
                       lnex_ld_avg_avgimpact 
                                           0 
                      lnex_ld_avg_insthindex 
                                           0 
                                ex_ld_bin_gs 
                                           0 
                              ex_ld_ratio_gs 
                                           0 
                             ex_ld_bin_sameC 
                                         

In [7]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [8]:
# data <- data %>% filter(!is.na(mean_career_age))
# data <- data %>% filter(!is.na(frac_hype_words))
# data <- data %>% filter(!is.na(source_hindex))
# data <- data %>% filter(!is.na(open_access))
# dim(data)

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$Facility <- as.factor(data$Facility)

In [11]:
data$CoType <- factor(data$CoType)
data <- within(data, CoType <- relevel(CoType, ref = 'Service'))
data$paper_type <- factor(data$paper_type)
data <- within(data, paper_type <- relevel(paper_type, ref = 'review'))
data$text_fac_scientist <- factor(data$text_fac_scientist)
data <- within(data, text_fac_scientist <- relevel(text_fac_scientist, ref = 'NonStaffPart'))
data$fac_scientist_lead_text <- factor(data$fac_scientist_lead_text)
data <- within(data, fac_scientist_lead_text <- relevel(fac_scientist_lead_text, ref = 'NonStaffLead'))
data$open_access <- factor(data$open_access)
data <- within(data, open_access <- relevel(open_access, ref = 'False'))
data$SDG <- factor(data$SDG)
data <- within(data, SDG <- relevel(SDG, ref = 'False'))
data$ex_ld_bin_sameC <- factor(data$ex_ld_bin_sameC)
data <- within(data, ex_ld_bin_sameC <- relevel(ex_ld_bin_sameC, ref = 'NonSame'))
data$ex_ld_bin_gs <- factor(data$ex_ld_bin_gs)
data <- within(data, ex_ld_bin_gs <- relevel(ex_ld_bin_gs, ref = 'GlobalSouth'))
data$ex_ld_max_before_year_with_ih_bin <- factor(data$ex_ld_max_before_year_with_ih_bin)
data <- within(data, ex_ld_max_before_year_with_ih_bin <- relevel(ex_ld_max_before_year_with_ih_bin, ref = 'False'))
data$ex_ld_max_before_year_participation_bin <- factor(data$ex_ld_max_before_year_participation_bin)
data <- within(data, ex_ld_max_before_year_participation_bin <- relevel(ex_ld_max_before_year_participation_bin, ref = 'False'))
data$ex_ld_max_before_year_co_lead_bin <- factor(data$ex_ld_max_before_year_co_lead_bin)
data <- within(data, ex_ld_max_before_year_co_lead_bin <- relevel(ex_ld_max_before_year_co_lead_bin, ref = 'False'))
data$international <- factor(data$international)
data <- within(data, international <- relevel(international, ref = 'domestic'))

In [12]:
paper_level <- "lnnum_author + international + lnnum_reference + num_fac + SDG + lnmean_career_age"
ex_controls <- "lnex_ld_avg_avgimpact + lnex_ld_avg_insthindex + ex_ld_bin_gs + ex_ld_bin_sameC + knowledge_proximity_mean"
moderating <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_with_ih_bin"
moderating2 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_participation_bin"
moderating3 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_co_lead_bin"
disciplines <- "Agricultural.and.Biological.Sciences + Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [13]:
paper_vars <- c("lnnum_author", "international", "lnnum_reference", "num_fac", "SDG", "lnmean_career_age")
ex_vars <- c("lnex_ld_avg_avgimpact", "lnex_ld_avg_insthindex", "ex_ld_bin_gs", "ex_ld_bin_sameC", "knowledge_proximity_mean")
moderating_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_with_ih_bin")
moderating2_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_participation_bin")
moderating3_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_co_lead_bin")
disciplines_vars <- c("Agricultural.and.Biological.Sciences", "Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

# H1:With > Without

In [14]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_total_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ],
                         family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_total_bin)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.072262   0.042679   1.693161
lnnum_author                                  0.005737   0.030202   0.189948
internationalinternational                   -0.047471   0.020000  -2.373577
lnnum_reference                               0.057475   0.025321   2.269868
num_fac                                       0.005753   0.023409   0.245758
SDGTrue                                       0.100131   0.021151   4.734092
lnmean_career_age                             0.082336   0.039867   2.065271
lnex_ld_avg_avgimpact                        -0.304499   0.028757 -10.588569
lnex_ld_avg_insthindex                       -0.081986   0.023079  -3.552360
ex_ld_bin_gsGlobalNorth                     

In [15]:
# 每组 reg_class 的平均预测概率
# pred_bin <- avg_predictions(model_total_bin, variables = "text_fac_scientist")
# pred_bin

In [16]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_pred.csv")
# write.csv(pred_bin, fname, row.names = FALSE)

In [17]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.072$^{*}$\\   
                                                       & (0.043)\\   
   lnnum\_author                                       & 0.006\\   
                                                       & (0.030)\\   
   internationalinternational                          & -0.048$^{**}$\\   
                                                       & (0.020)\\   
   lnnum\_reference                                    & 0.058$^{**}$\\   
                                                       & (0.025)\\   
   num\_fac                                            & 0.006\\   
                                                       & (0.023)\\   
   SDGTrue                    

In [18]:
# margins_eff_bin <- avg_comparisons(model_total_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_bin

In [19]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_comp_ratio.csv")
# write.csv(margins_eff_bin, fname, row.names = FALSE)

# H1 different disciplines

In [20]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_ps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ],
                      family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_ps_bin)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 253,547
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error   z value
text_fac_scientistStaffPart                   0.067093   0.036070  1.860058
lnnum_author                                 -0.098963   0.023584 -4.196216
internationalinternational                   -0.070231   0.022295 -3.150129
lnnum_reference                               0.087503   0.027362  3.197933
num_fac                                       0.040977   0.025227  1.624340
SDGTrue                                       0.050226   0.023400  2.146405
lnmean_career_age                             0.013716   0.039771  0.344868
lnex_ld_avg_avgimpact                        -0.239504   0.031577 -7.584800
lnex_ld_avg_insthindex                       -0.051280   0.021207 -2.418034
ex_ld_bin_gsGlobalNorth                       0.422515

In [21]:
# margins_eff_ps_bin <- avg_comparisons(model_ps_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_ps_bin

In [22]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ps_comp_ratio.csv")
# write.csv(margins_eff_ps_bin, fname, row.names = FALSE)

In [23]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.067$^{*}$\\   
                                                       & (0.036)\\   
   lnnum\_author                                       & -0.099$^{***}$\\   
                                                       & (0.024)\\   
   internationalinternational                          & -0.070$^{***}$\\   
                                                       & (0.022)\\   
   lnnum\_reference                                    & 0.087$^{***}$\\   
                                                       & (0.027)\\   
   num\_fac                                            & 0.041\\   
                                                       & (0.025)\\   
   SDGTrue         

In [24]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_ls_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ],
                      family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_ls_bin)

NOTE: 5/1 fixed-effects (11 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 77,580
Fixed-effects: Facility: 66,  PublishedYear: 48
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.176357   0.070604   2.497832
lnnum_author                                  0.313270   0.036744   8.525778
internationalinternational                    0.064215   0.016754   3.832807
lnnum_reference                               0.061268   0.030327   2.020256
num_fac                                      -0.112997   0.034956  -3.232604
SDGTrue                                       0.169404   0.028882   5.865424
lnmean_career_age                             0.210953   0.046733   4.514052
lnex_ld_avg_avgimpact                        -0.373767   0.033801 -11.057697
lnex_ld_avg_insthindex                       -0.139056   0.033543  -4.145655
ex_ld_bin_gsGlobalNorth                      

In [25]:
# margins_eff_ls_bin <- avg_comparisons(model_ls_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_ls_bin

In [26]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ls_comp_ratio.csv")
# write.csv(margins_eff_ls_bin, fname, row.names = FALSE)

In [27]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.176$^{**}$\\   
                                                       & (0.071)\\   
   lnnum\_author                                       & 0.313$^{***}$\\   
                                                       & (0.037)\\   
   internationalinternational                          & 0.064$^{***}$\\   
                                                       & (0.017)\\   
   lnnum\_reference                                    & 0.061$^{**}$\\   
                                                       & (0.030)\\   
   num\_fac                                            & -0.113$^{***}$\\   
                                                       & (0.035)\\   
   SDGTrue  

In [28]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_hs_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ],
                      family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_hs_bin)

NOTE: 10/6 fixed-effects (20 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 33,397
Fixed-effects: Facility: 60,  PublishedYear: 40
Standard-errors: Clustered (Facility & PublishedYear) 
                                               Estimate Std. Error     z value
text_fac_scientistStaffPart                    0.419388   0.086084    4.871839
lnnum_author                                   0.350733   0.054176    6.473985
internationalinternational                     0.053615   0.019991    2.682025
lnnum_reference                                0.090192   0.043328    2.081602
num_fac                                       -0.134435   0.038330   -3.507287
SDGTrue                                        0.227529   0.032271    7.050613
lnmean_career_age                              0.241564   0.064061    3.770864
lnex_ld_avg_avgimpact                         -0.354195   0.039175   -9.041315
lnex_ld_avg_insthindex                        -0.124111   0.037588   -3.301877
ex_ld_bin_gsGlobalNorth  

In [29]:
# margins_eff_hs_bin <- avg_comparisons(model_hs_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_hs_bin

In [30]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_hs_comp_ratio.csv")
# write.csv(margins_eff_hs_bin, fname, row.names = FALSE)

In [31]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.419$^{***}$\\   
                                                       & (0.086)\\   
   lnnum\_author                                       & 0.351$^{***}$\\   
                                                       & (0.054)\\   
   internationalinternational                          & 0.054$^{***}$\\   
                                                       & (0.020)\\   
   lnnum\_reference                                    & 0.090$^{**}$\\   
                                                       & (0.043)\\   
   num\_fac                                            & -0.134$^{***}$\\   
                                                       & (0.038)\\   
   SDGTrue 

In [32]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_nps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ],
                       family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_nps_bin)

NOTE: 4/6 fixed-effects (19 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 41,285
Fixed-effects: Facility: 53,  PublishedYear: 38
Standard-errors: Clustered (Facility & PublishedYear) 
                                               Estimate Std. Error    z value
text_fac_scientistStaffPart                    0.362755   0.095656   3.792309
lnnum_author                                   0.415265   0.039116  10.616209
internationalinternational                     0.082774   0.005365  15.428901
lnnum_reference                               -0.123873   0.037329  -3.318440
num_fac                                       -0.187107   0.036764  -5.089377
SDGTrue                                        0.268680   0.040123   6.696468
lnmean_career_age                              0.344457   0.046464   7.413466
lnex_ld_avg_avgimpact                         -0.518251   0.055429  -9.349762
lnex_ld_avg_insthindex                        -0.197943   0.044324  -4.465796
ex_ld_bin_gsGlobalNorth            

In [33]:
# margins_eff_nps_bin <- avg_comparisons(model_nps_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_nps_bin

In [34]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_nps_comp_ratio.csv")
# write.csv(margins_eff_nps_bin, fname, row.names = FALSE)

In [35]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.363$^{***}$\\   
                                                       & (0.096)\\   
   lnnum\_author                                       & 0.415$^{***}$\\   
                                                       & (0.039)\\   
   internationalinternational                          & 0.083$^{***}$\\   
                                                       & (0.005)\\   
   lnnum\_reference                                    & -0.124$^{***}$\\   
                                                       & (0.037)\\   
   num\_fac                                            & -0.187$^{***}$\\   
                                                       & (0.037)\\   
   SDGTru

# H2: Collaboration > Participation

In [36]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_total <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ],
                     family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_total)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.105164   0.047323   2.222250
CoTypeParticipation                           0.002214   0.046086   0.048034
lnnum_author                                  0.010312   0.029628   0.348051
internationalinternational                   -0.047690   0.019895  -2.397101
lnnum_reference                               0.058242   0.025377   2.295073
num_fac                                       0.005486   0.023511   0.233333
SDGTrue                                       0.099661   0.021107   4.721690
lnmean_career_age                             0.079383   0.040571   1.956666
lnex_ld_avg_avgimpact                        -0.305114   0.028665 -10.644055
lnex_ld_avg_insthindex                      

In [37]:
# # 每组 reg_class 的平均预测概率
# pred <- avg_predictions(model_total, variables = "CoType")
# pred

In [38]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_pred.csv")
# write.csv(pred, fname, row.names = FALSE)

In [39]:
# # 每组 reg_class 的平均预测概率
# margins_eff <- avg_comparisons(model_total, variables = "CoType", comparison = 'ratio')
# margins_eff

In [40]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_comp_ratio.csv")
# write.csv(margins_eff, fname, row.names = FALSE)

In [41]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.105$^{**}$\\   
                                                       & (0.047)\\   
   CoTypeParticipation                                 & 0.002\\   
                                                       & (0.046)\\   
   lnnum\_author                                       & 0.010\\   
                                                       & (0.030)\\   
   internationalinternational                          & -0.048$^{**}$\\   
                                                       & (0.020)\\   
   lnnum\_reference                                    & 0.058$^{**}$\\   
                                                       & (0.025)\\   
   num\_fac                  

# H2 Discipline

In [42]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_ps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ],
                  family = binomial("logit"), vcov = ~Facility + PublishedYear)
summary(model_ps)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 253,547
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error   z value
CoTypeCollaboration                           0.095878   0.041153  2.329782
CoTypeParticipation                           0.004349   0.041768  0.104112
lnnum_author                                 -0.094080   0.023571 -3.991377
internationalinternational                   -0.070422   0.022197 -3.172594
lnnum_reference                               0.088162   0.027387  3.219136
num_fac                                       0.040697   0.025374  1.603904
SDGTrue                                       0.049858   0.023338  2.136349
lnmean_career_age                             0.010956   0.040455  0.270817
lnex_ld_avg_avgimpact                        -0.240258   0.031485 -7.630857
lnex_ld_avg_insthindex                       -0.050340

In [43]:
# # 每组 reg_class 的平均预测概率
# margins_eff_ps <- avg_comparisons(model_ps, variables = "CoType", comparison = 'ratio')
# margins_eff_ps

In [44]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ps_comp_ratio.csv")
# write.csv(margins_eff_ps, fname, row.names = FALSE)

In [45]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.096$^{**}$\\   
                                                       & (0.041)\\   
   CoTypeParticipation                                 & 0.004\\   
                                                       & (0.042)\\   
   lnnum\_author                                       & -0.094$^{***}$\\   
                                                       & (0.024)\\   
   internationalinternational                          & -0.070$^{***}$\\   
                                                       & (0.022)\\   
   lnnum\_reference                                    & 0.088$^{***}$\\   
                                                       & (0.027)\\   
   num\_fac       

In [46]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_ls <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ],
                  family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_ls)

NOTE: 5/1 fixed-effects (11 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 77,580
Fixed-effects: Facility: 66,  PublishedYear: 48
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.162922   0.071102   2.291388
CoTypeParticipation                           0.209086   0.077856   2.685559
lnnum_author                                  0.313050   0.036640   8.543928
internationalinternational                    0.064128   0.016770   3.824084
lnnum_reference                               0.061079   0.030266   2.018051
num_fac                                      -0.112771   0.034854  -3.235526
SDGTrue                                       0.169453   0.028892   5.865132
lnmean_career_age                             0.211941   0.046733   4.535168
lnex_ld_avg_avgimpact                        -0.373645   0.033736 -11.075708
lnex_ld_avg_insthindex                       

In [47]:
# # 每组 reg_class 的平均预测概率
# margins_eff_ls <- avg_comparisons(model_ls, variables = "CoType", comparison = 'ratio')
# margins_eff_ls

In [48]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ls_comp_ratio.csv")
# write.csv(margins_eff_ls, fname, row.names = FALSE)

In [49]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.163$^{**}$\\   
                                                       & (0.071)\\   
   CoTypeParticipation                                 & 0.209$^{***}$\\   
                                                       & (0.078)\\   
   lnnum\_author                                       & 0.313$^{***}$\\   
                                                       & (0.037)\\   
   internationalinternational                          & 0.064$^{***}$\\   
                                                       & (0.017)\\   
   lnnum\_reference                                    & 0.061$^{**}$\\   
                                                       & (0.030)\\   
   num\_fac  

In [50]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_hs <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ],
                  family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_hs)

NOTE: 10/6 fixed-effects (20 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 33,397
Fixed-effects: Facility: 60,  PublishedYear: 40
Standard-errors: Clustered (Facility & PublishedYear) 
                                               Estimate Std. Error     z value
CoTypeCollaboration                            0.388760   0.096163    4.042721
CoTypeParticipation                            0.486486   0.116546    4.174193
lnnum_author                                   0.349545   0.053903    6.484661
internationalinternational                     0.053370   0.020189    2.643484
lnnum_reference                                0.090174   0.043366    2.079350
num_fac                                       -0.134458   0.038476   -3.494587
SDGTrue                                        0.227767   0.032183    7.077279
lnmean_career_age                              0.243342   0.064962    3.745928
lnex_ld_avg_avgimpact                         -0.353787   0.038928   -9.088335
lnex_ld_avg_insthindex   

In [51]:
# # 每组 reg_class 的平均预测概率
# margins_eff_hs <- avg_comparisons(model_hs, variables = "CoType", comparison = 'ratio')
# margins_eff_hs

In [52]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_hs_comp_ratio.csv")
# write.csv(margins_eff_hs, fname, row.names = FALSE)

In [53]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.389$^{***}$\\   
                                                       & (0.096)\\   
   CoTypeParticipation                                 & 0.486$^{***}$\\   
                                                       & (0.117)\\   
   lnnum\_author                                       & 0.350$^{***}$\\   
                                                       & (0.054)\\   
   internationalinternational                          & 0.053$^{***}$\\   
                                                       & (0.020)\\   
   lnnum\_reference                                    & 0.090$^{**}$\\   
                                                       & (0.043)\\   
   num\_fac 

In [54]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_nps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ],
                   family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_nps)

NOTE: 4/6 fixed-effects (19 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 41,285
Fixed-effects: Facility: 53,  PublishedYear: 38
Standard-errors: Clustered (Facility & PublishedYear) 
                                               Estimate Std. Error    z value
CoTypeCollaboration                            0.334116   0.103016   3.243333
CoTypeParticipation                            0.421273   0.105612   3.988883
lnnum_author                                   0.414747   0.038849  10.675827
internationalinternational                     0.082908   0.005464  15.173462
lnnum_reference                               -0.124189   0.037319  -3.327753
num_fac                                       -0.186837   0.036814  -5.075175
SDGTrue                                        0.268909   0.040050   6.714300
lnmean_career_age                              0.345954   0.046466   7.445286
lnex_ld_avg_avgimpact                         -0.518264   0.055369  -9.360125
lnex_ld_avg_insthindex             

In [55]:
# # 每组 reg_class 的平均预测概率
# margins_eff_nps <- avg_comparisons(model_nps, variables = "CoType", comparison = 'ratio')
# margins_eff_nps

In [56]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_nps_comp_ratio.csv")
# write.csv(margins_eff_nps, fname, row.names = FALSE)

In [57]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.334$^{***}$\\   
                                                       & (0.103)\\   
   CoTypeParticipation                                 & 0.421$^{***}$\\   
                                                       & (0.106)\\   
   lnnum\_author                                       & 0.415$^{***}$\\   
                                                       & (0.039)\\   
   internationalinternational                          & 0.083$^{***}$\\   
                                                       & (0.005)\\   
   lnnum\_reference                                    & -0.124$^{***}$\\   
                                                       & (0.037)\\   
   num\_fa

# H3: Too much will suppress

# H3a: Participation too much not good

In [58]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ ratio_fac_scientist + I(ratio_fac_scientist^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_h3_pratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ],
                         family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_h3_pratio)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error    z value
ratio_fac_scientist                           0.313398   0.213404   1.468564
I(ratio_fac_scientist^2)                     -0.151758   0.238400  -0.636569
lnnum_author                                  0.011735   0.029529   0.397415
internationalinternational                   -0.046648   0.019869  -2.347841
lnnum_reference                               0.058730   0.025358   2.316046
num_fac                                       0.005042   0.023817   0.211691
SDGTrue                                       0.099749   0.021145   4.717341
lnmean_career_age                             0.081262   0.040139   2.024540
lnex_ld_avg_avgimpact                        -0.304830   0.028736 -10.608050
lnex_ld_avg_insthindex                      

In [59]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_pratio, condition = "ratio_fac_scientist", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("ratio_fac_scientist", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_pratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

In [60]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_pratio,
                           keep = c("ratio_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   ratio\_fac\_scientist                               & 0.313\\   
                                                       & (0.213)\\   
   ratio\_fac\_scientist square                        & -0.152\\   
                                                       & (0.238)\\   
   lnnum\_author                                       & 0.012\\   
                                                       & (0.029)\\   
   internationalinternational                          & -0.047$^{**}$\\   
                                                       & (0.020)\\   
   lnnum\_reference                                    & 0.059$^{**}$\\   
                                                       & (0.025)\\   
   num\_fac                        

# H3b: Lead too much not good

In [61]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ fac_scientist_lead_ratio + I(fac_scientist_lead_ratio^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_h3_lratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$CoType_Service==0)&(data$knowledge_proximity_mean>0), ],
                         family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_h3_lratio)

NOTE: 3/1 fixed-effects (25 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 78,508
Fixed-effects: Facility: 70,  PublishedYear: 50
Standard-errors: Clustered (Facility & PublishedYear) 
                                              Estimate Std. Error   z value
fac_scientist_lead_ratio                      0.481085   0.199852  2.407207
I(fac_scientist_lead_ratio^2)                -0.444326   0.238818 -1.860522
lnnum_author                                 -0.112088   0.056372 -1.988348
internationalinternational                   -0.124689   0.039114 -3.187864
lnnum_reference                               0.102381   0.025014  4.092880
num_fac                                       0.031603   0.030666  1.030541
SDGTrue                                       0.099540   0.026848  3.707521
lnmean_career_age                            -0.064868   0.046569 -1.392958
lnex_ld_avg_avgimpact                        -0.181606   0.038667 -4.696675
lnex_ld_avg_insthindex                       -0.041286 

In [62]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_lratio, condition = "fac_scientist_lead_ratio", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("fac_scientist_lead_ratio", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_lratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

In [63]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_lratio,
                           keep = c("fac_scientist_lead_ratio", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_scientist\_lead\_ratio                         & 0.481$^{**}$\\   
                                                       & (0.200)\\   
   fac\_scientist\_lead\_ratio square                  & -0.444$^{*}$\\   
                                                       & (0.239)\\   
   lnnum\_author                                       & -0.112$^{**}$\\   
                                                       & (0.056)\\   
   internationalinternational                          & -0.125$^{***}$\\   
                                                       & (0.039)\\   
   lnnum\_reference                                    & 0.102$^{***}$\\   
                                                       & (0.025)\\   
   num\_fac 

# Moderating

In [64]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnex_ld_avg_before_year_prod_fac  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ],
                          family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_pre_facpub)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                                      Estimate Std. Error
CoTypeCollaboration                                   0.133257   0.058854
CoTypeParticipation                                   0.159713   0.085306
lnex_ld_avg_before_year_prod_fac                     -0.043231   0.017109
lnnum_author                                          0.011758   0.029468
internationalinternational                           -0.048024   0.019818
lnnum_reference                                       0.058361   0.025449
num_fac                                               0.007951   0.023461
SDGTrue                                               0.099434   0.021102
lnmean_career_age                                     0.078590   0.040631
lnex_ld_avg_avgimpact                                -0.304816   0.028612


In [65]:
# # 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
# min_val <- min(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# max_val <- max(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# # 2. 运行估计
# res_pre_facpub <- avg_comparisons(
#     model_pre_facpub,
#     variables = "CoType",
#     comparison = "ratio",
#     newdata = datagrid(
#     model = model_pre_facpub,
#     lnex_ld_avg_before_year_prod_fac = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
#   ),
#     by = "lnex_ld_avg_before_year_prod_fac"
# )
# res_pre_facpub

In [66]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_facpub.csv")
# write.csv(res_pre_facpub, fname, row.names = FALSE)

In [67]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                        & novel\_uzzi\_bin\\    
   Model:                                                                     & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                        & 0.133$^{**}$\\   
                                                                              & (0.059)\\   
   CoTypeParticipation                                                        & 0.160$^{*}$\\   
                                                                              & (0.085)\\   
   lnex\_ld\_avg\_before\_year\_prod\_fac                                     & -0.043$^{**}$\\   
                                                                              & (0.017)\\   
   lnnum\_author                                                              & 0.012\\   
                               

In [68]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*ex_ld_max_before_year_with_ih_bin  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | Facility + PublishedYear")
)
model_pre_withih <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ],
                          family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_pre_withih)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                                           Estimate Std. Error
CoTypeCollaboration                                        0.246669   0.054852
CoTypeParticipation                                        0.260967   0.072081
ex_ld_max_before_year_with_ih_binTrue                      0.223709   0.026364
lnnum_author                                               0.010997   0.029612
internationalinternational                                -0.048976   0.019732
lnnum_reference                                            0.058895   0.025416
num_fac                                                    0.006420   0.023572
SDGTrue                                                    0.099551   0.021118
lnmean_career_age                                          0.077906   0.040724
lnex_ld_avg_avgimpact   

In [69]:
# # 每组 reg_class 的平均预测概率
# res_pre_withih <- avg_comparisons(model_pre_withih, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_with_ih_bin')
# res_pre_withih

In [70]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_withih.csv")
# write.csv(res_pre_withih, fname, row.names = FALSE)

In [71]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_withih,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                               & novel\_uzzi\_bin\\    
   Model:                                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                               & 0.247$^{***}$\\   
                                                                                     & (0.055)\\   
   CoTypeParticipation                                                               & 0.261$^{***}$\\   
                                                                                     & (0.072)\\   
   ex\_ld\_max\_before\_year\_with\_ih\_binTrue                                      & 0.224$^{***}$\\   
                                                                                     & (0.026)\\   
   lnnum\_author                                               

In [72]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*ex_ld_max_before_year_participation_bin  + ", paper_level, "+", ex_controls, "+", moderating2, "+",disciplines, " | Facility + PublishedYear")
)
model_pre_partic <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ],
                          family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_pre_partic)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                                                 Estimate
CoTypeCollaboration                                              0.164030
CoTypeParticipation                                              0.183166
ex_ld_max_before_year_participation_binTrue                      0.115545
lnnum_author                                                     0.017910
internationalinternational                                      -0.045972
lnnum_reference                                                  0.060669
num_fac                                                          0.003811
SDGTrue                                                          0.100329
lnmean_career_age                                                0.087010
lnex_ld_avg_avgimpact                                           -0.305043


In [73]:
# # 每组 reg_class 的平均预测概率
# res_pre_partic <- avg_comparisons(model_pre_partic, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_participation_bin')
# res_pre_partic

In [74]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_partic.csv")
# write.csv(res_pre_partic, fname, row.names = FALSE)

In [75]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_partic,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                                   & novel\_uzzi\_bin\\    
   Model:                                                                                & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                                   & 0.164$^{***}$\\   
                                                                                         & (0.059)\\   
   CoTypeParticipation                                                                   & 0.183$^{***}$\\   
                                                                                         & (0.060)\\   
   lnnum\_author                                                                         & 0.018\\   
                                                                                         & (0.030)\\   
   internationalinternational          

In [76]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*ex_ld_max_before_year_co_lead_bin  + ", paper_level, "+", ex_controls, "+", moderating3, "+",disciplines, " | Facility + PublishedYear")
)
model_pre_co_lead <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = ~Facility + PublishedYear,)
summary(model_pre_co_lead)

NOTE: 2/0 fixed-effects (28 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,851
Fixed-effects: Facility: 74,  PublishedYear: 51
Standard-errors: Clustered (Facility & PublishedYear) 
                                                           Estimate Std. Error
CoTypeCollaboration                                        0.217241   0.051335
CoTypeParticipation                                        0.232294   0.072148
ex_ld_max_before_year_co_lead_binTrue                      0.221637   0.026483
lnnum_author                                               0.010601   0.029653
internationalinternational                                -0.048491   0.019569
lnnum_reference                                            0.058114   0.025378
num_fac                                                    0.006286   0.023447
SDGTrue                                                    0.099453   0.021115
lnmean_career_age                                          0.076718   0.040876
lnex_ld_avg_avgimpact   

In [77]:
# # 每组 reg_class 的平均预测概率
# res_pre_co_lead <- avg_comparisons(model_pre_co_lead, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_co_lead_bin')
# res_pre_co_lead

In [78]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_co_lead.csv")
# write.csv(res_pre_co_lead, fname, row.names = FALSE)

In [79]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_co_lead,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           vcov = ~Facility + PublishedYear,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                               & novel\_uzzi\_bin\\    
   Model:                                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                               & 0.217$^{***}$\\   
                                                                                     & (0.051)\\   
   CoTypeParticipation                                                               & 0.232$^{***}$\\   
                                                                                     & (0.072)\\   
   lnnum\_author                                                                     & 0.011\\   
                                                                                     & (0.030)\\   
   internationalinternational                                          

# 补充一个更deep的point，曾经开展过“Co-lead”,后续合作/参与的收益受损更严重

In [88]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", disciplines, " | PublishedYear")
)
model1 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model1)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.168124   0.010601  15.859520
CoTypeParticipation                           0.047151   0.014212   3.317634
Arts.and.Humanities                           1.837644   0.072240  25.438148
Biochemistry..Genetics.and.Molecular.Biology  0.498277   0.012207  40.818163
Business..Management.and.Accounting          -0.428431   0.087633  -4.888910
Chemical.Engineering                          0.131064   0.020722   6.324869
Chemistry                                     0.458689   0.010925  41.986380
Computer.Science                              0.896830   0.039192  22.883139
Decision.Sciences                             1.233745   0.187252   6.588695
Dentistry                                     1.993278   0.134826  14.

In [89]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+",disciplines, " | PublishedYear")
)
model2 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model2)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.176029   0.010963  16.055991
CoTypeParticipation                           0.095087   0.014820   6.416025
lnnum_author                                 -0.130941   0.007103 -18.433963
internationalinternational                   -0.007958   0.008684  -0.916399
lnnum_reference                              -0.005923   0.008397  -0.705350
num_fac                                       0.050942   0.006548   7.779950
SDGTrue                                       0.087651   0.008086  10.840337
lnmean_career_age                            -0.007336   0.012674  -0.578814
Arts.and.Humanities                           1.805187   0.072180  25.009688
Biochemistry..Genetics.and.Molecular.Biology  0.488323   0.012268  39.

In [90]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
model3 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model3)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.143687   0.011097  12.948869
CoTypeParticipation                           0.047011   0.014969   3.140640
lnnum_author                                 -0.068658   0.007116  -9.648419
internationalinternational                   -0.031089   0.009559  -3.252299
lnnum_reference                               0.061520   0.008603   7.150632
num_fac                                       0.059498   0.006605   9.007958
SDGTrue                                       0.092371   0.008114  11.384687
lnmean_career_age                             0.033050   0.012852   2.571617
lnex_ld_avg_avgimpact                        -0.264828   0.007259 -36.483771
lnex_ld_avg_insthindex                       -0.049529   0.007468  -6.

In [91]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model4 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model4)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.099926   0.011245   8.886446
CoTypeParticipation                           0.010299   0.015044   0.684582
lnnum_author                                 -0.087020   0.007328 -11.874570
internationalinternational                   -0.045118   0.009576  -4.711696
lnnum_reference                               0.059057   0.008633   6.841129
num_fac                                       0.062924   0.006715   9.369983
SDGTrue                                       0.093974   0.008128  11.561174
lnmean_career_age                             0.012005   0.013072   0.918354
lnex_ld_avg_avgimpact                        -0.269872   0.007355 -36.690437
lnex_ld_avg_insthindex                       -0.051397   0.007510  -6.

In [92]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model1, model2, model3, model4,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + r2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.168$^{***}$  & 0.176$^{***}$  & 0.144$^{***}$  & 0.100$^{***}$\\                                                           & (0.011)        & (0.011)        & (0.011)        & (0.011)\\       CoTypeParticipation                                 & 0.047$^{***}$  & 0.095$^{***}$  & 0.047$^{***}$  & 0.010\\                                                           & (0.014)        & (0.015)        & (0.015)        & (0.015)\\       Arts.and.Humanities                                 & 1.84$^{***}$   & 1.81$^{***}$   & 1.76$^{***}$   & 1.74$^{***}$\\                                                           & (0.072)        & (0.072)        